# DeepLab: Atrous Convolution for Semantic Segmentation

The **DeepLab** family (v1, v2, v3, v3+) is a series of semantic-segmentation models from Google. Its central problem is a tension inherent to CNN classifiers repurposed for dense prediction:

- **Downsampling** (strided convolutions and pooling) grows the receptive field and builds semantic abstraction, but it **destroys spatial resolution**. A `ResNet` backbone reduces a `513x513` input to roughly `17x17`.
- Semantic segmentation needs a **per-pixel** label at (near) the input resolution, with sharp object boundaries.

DeepLab's answer is **atrous (dilated) convolution**, which enlarges the receptive field **without** downsampling and **without** adding parameters, plus **atrous spatial pyramid pooling (ASPP)** to capture objects at multiple scales.

## Atrous (dilated) convolution

An atrous convolution inserts `rate - 1` zeros ("holes", *a trous* in French) between the weights of a standard kernel. For a 1-D signal it computes:

$$y[i] = \sum_{k} x[i + r\cdot k]\, w[k]$$

where `r` is the **atrous rate**. With `r = 1` it is an ordinary convolution. With `r > 1` the kernel "sees" a wider input span while keeping the same number of weights and the same output resolution.

**Effective kernel size** of a `k x k` kernel at rate `r` is:

$$k_e = k + (k - 1)(r - 1)$$

so a `3x3` kernel at `rate=2` covers a `5x5` field, at `rate=4` covers `9x9`, etc. — all with just 9 weights.

```python
import torch.nn as nn

# 3x3 conv with dilation=2 -> effective 5x5 receptive field, resolution preserved (padding=2)
conv = nn.Conv2d(in_channels=256, out_channels=256,
                 kernel_size=3, padding=2, dilation=2)
```

In DeepLab the deep stages of the backbone replace stride with atrous convolution so the **output stride** (input size / feature size) stays at 16 or 8 instead of 32, dramatically increasing the density of the final feature map.

## ASPP: Atrous Spatial Pyramid Pooling

Objects appear at many scales, so a single receptive field is not enough. **ASPP** probes the feature map with several parallel atrous convolutions at different rates, then fuses them.

A typical DeepLabv3 ASPP module (at output stride 16) contains:

- a `1x1` convolution,
- three `3x3` atrous convolutions at rates `(6, 12, 18)`,
- an **image-level** branch: global average pooling -> `1x1` conv -> bilinear upsample back to the feature size (gives global context).

The five branches are concatenated along the channel dimension and passed through a final `1x1` conv. This multi-rate sampling captures both fine and coarse context in one shot.

```
feature map
   |--- 1x1 conv ----------------+
   |--- 3x3 conv  rate=6 --------+
   |--- 3x3 conv  rate=12 -------+--> concat --> 1x1 conv --> output
   |--- 3x3 conv  rate=18 -------+
   |--- global avg pool -> 1x1 -> upsample +
```

## Evolution: v1 -> v3+

| Version | Key additions |
|---|---|
| **DeepLabv1** | Atrous convolution + a fully-connected **CRF** (conditional random field) as a post-processing step to sharpen boundaries. |
| **DeepLabv2** | Introduces **ASPP** (multi-rate atrous); still uses dense CRF post-processing. |
| **DeepLabv3** | Improved ASPP (adds the image-level/global-pooling branch and batch norm); **drops the CRF** — the denser features make it unnecessary. |
| **DeepLabv3+** | Adds a lightweight **decoder** (encoder-decoder structure): the ASPP output is upsampled and fused with an early low-level backbone feature to recover sharp boundaries, instead of a single naive 16x bilinear upsample. Often pairs with an **Xception** backbone and depthwise-separable atrous convolutions. |

The **CRF** in v1/v2 models pairwise pixel relationships (color/position similarity) to refine the coarse network output along true edges; from v3 onward the architecture itself produces boundaries clean enough that the CRF was removed.

## Using torchvision's DeepLabv3

`torchvision` ships DeepLabv3 with ResNet-50/101 and MobileNetV3 backbones, pretrained on a COCO/VOC subset (21 classes, including background).

```python
import torch
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights).eval()

preprocess = weights.transforms()          # resize + normalize, returns a tensor

from PIL import Image
img = Image.open("street.jpg").convert("RGB")
batch = preprocess(img).unsqueeze(0)        # [1, 3, H, W]

with torch.no_grad():
    output = model(batch)["out"]            # dict with key 'out': [1, 21, H, W] logits

mask = output.argmax(1).squeeze(0)          # [H, W] per-pixel class index
print(weights.meta["categories"])           # class names for each channel
```

The model returns a **dict** whose `"out"` value holds the segmentation logits (and `"aux"` if the auxiliary classifier is enabled). Note the output is already upsampled to the input spatial size.

## References

- Chen et al., *Semantic Image Segmentation with Deep Convolutional Nets and Fully Connected CRFs* (DeepLabv1), 2015 — https://arxiv.org/abs/1412.7062
- Chen et al., *DeepLab: Atrous Convolution, ASPP, and Fully Connected CRFs* (v2), 2017 — https://arxiv.org/abs/1606.00915
- Chen et al., *Rethinking Atrous Convolution for Semantic Image Segmentation* (DeepLabv3), 2017 — https://arxiv.org/abs/1706.05587
- Chen et al., *Encoder-Decoder with Atrous Separable Convolution* (DeepLabv3+), 2018 — https://arxiv.org/abs/1802.02611
- torchvision segmentation models — https://pytorch.org/vision/stable/models.html#semantic-segmentation